## COSMOS Lab 5: Linear Regression and KNN Regression - Advanced Version

**UCLA COSMOS Cluster 10: IoT + Data Science**

This notebook is part of the COSMOS data science lab sequence. Work through the code cells in order and discuss the written prompts with your group.


## Learning Goals

By the end of this lab, you should be able to:

* Review `numpy` including 2-D arrays and array reshaping.
* Use `matplotlib` to make clear exploratory plots.
* Fit simple and multiple linear regression models.
* Use `scikit-learn` with a real IoT sensor dataset.
* Compare linear regression and KNN regression for home appliance energy prediction.
* Explain why feature scaling matters for KNN.
* **Additional:** Write linear regression in matrix form and compute the closed-form solution.
* **Additional:** Explain what happens when the inverse does not exist and use the pseudoinverse.
* **Additional:** Interpret least squares as projecting onto the nearest plane/subspace.



## Table of Contents

* [1 - Review of numpy](#first-bullet)
* [2 - Plotting with matplotlib](#second-bullet)
* [3 - Simple Linear Regression](#third-bullet)
* [Additional - Matrix form, pseudoinverse, and projection](#additional-matrix-linear-regression)
* [4 - Building a model with `sklearn`](#fourth-bullet)
* [5 - Linear regression for appliance energy prediction](#fifth-bullet)
* [6 - K-nearest neighbors regression](#sixth-bullet)


In [ ]:
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import pandas as pd
import time
pd.set_option('display.width', 500)
pd.set_option('display.max_columns', 100)
pd.set_option('display.notebook_repr_html', True)
#import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
# Displays the plots for us.
%matplotlib inline

<a class="anchor" id="first-bullet"></a>
## 1 - Review of  the  `numpy` Python library

In lab1 we learned about the `numpy` library [(documentation)](http://www.numpy.org/) and its fast array structure, called the `numpy array`.

In [ ]:
# import numpy
import numpy as np

In [ ]:
# make an array
my_array = np.array([1,4,9,16])
my_array

In [ ]:
print(f'Size of my array: {my_array.size}, or length of my array: {len(my_array)}')
print (f'Shape of my array: {my_array.shape}')

#### Notice the way the shape appears in numpy arrays

- For a 1D array, .shape returns a tuple with 1 element (n,)
- For a 2D array, .shape returns a tuple with 2 elements (n,m)
- For a 3D array, .shape returns a tuple with 3 elements (n,m,p)

In [ ]:
# How to reshape a 1D array to a 2D
my_array.reshape(-1,2)

Numpy arrays support the same operations as lists! Below we slice and iterate.

In [ ]:
print("array[2:4]:", my_array[2:4]) # A slice of the array

# Iterate over the array
for ele in my_array:
    print("element:", ele)

Remember `numpy` gains a lot of its efficiency from being **strongly typed** (all elements are of the same type, such as integer or floating point). If the elements of an array are of a different type, `numpy` will force them into the same type (the longest in terms of bytes)

In [ ]:
mixed = np.array([1, 2.3, 'eleni', True])
print(type(1), type(2.3), type('eleni'), type(True))
mixed # all elements will become strings

Next, we push ahead to two-dimensional arrays and begin to dive into some of the deeper aspects of `numpy`.

In [ ]:
# create a 2d-array by handing a list of lists
my_array2d = np.array([ [1, 2, 3, 4],
                        [5, 6, 7, 8],
                        [9, 10, 11, 12]
])

my_array2d

### Array Slicing (a reminder...)

Numpy arrays can be sliced, and can be iterated over with loops.  Below is a schematic illustrating slicing two-dimensional arrays.
A two-dimensional NumPy array can be indexed by row and column using `array[row_slice, column_slice]`.
Notice that the list slicing syntax still works!  
`array[2:,3]` says "in the array, get rows 2 through the end, column 3]"  
`array[3,:]` says "in the array, get row 3, all columns".

### Pandas Slicing (a reminder...)

`.iloc` is by position (position is unique), `.loc` is by label (label is not unique)

In [ ]:
# Build a small IoT-style sensor table for pandas slicing practice.
sensor_log = pd.DataFrame({
    "timestamp": pd.date_range("2026-07-10 13:00", periods=15, freq="10min"),
    "room": ["kitchen", "bedroom", "office", "living"] * 3 + ["kitchen", "bedroom", "office"],
    "temperature_C": [22.1, 21.8, 23.0, 22.5, 22.4, 21.9, 23.2, 22.7, 22.8, 22.0, 23.1, 22.6, 22.9, 22.1, 23.3],
    "humidity_pct": [42, 45, 39, 41, 43, 46, 38, 40, 44, 45, 39, 41, 43, 44, 38],
    "motion_count": [0, 1, 0, 3, 4, 1, 0, 2, 5, 0, 1, 4, 3, 0, 2],
    "appliance_Wh": [35, 30, 25, 60, 95, 40, 30, 70, 110, 35, 45, 85, 75, 30, 65],
})

sensor_log

In [ ]:
# Get rows 10 to 13 using Python slicing style: exclusive of the end.
sensor_log.iloc[10:13]

In [ ]:
# Get columns 0 to 2 for all rows, then show the first few rows.
sensor_log.iloc[:, 0:3].head()

In [ ]:
# Get rows 10 to 13 and only columns 0 to 2.
sensor_log.iloc[10:13, 0:3]

In [ ]:
# Compare with .loc: label slicing is inclusive of the end label.
sensor_log.loc[10:13]

In [ ]:
# Select named columns by label for rows 5 to 10.
sensor_log.loc[5:10, ["timestamp", "room", "appliance_Wh"]]

### Python Trick of the Day

In [ ]:
import re
names = ['mayday','springday','horseday','june']

In [ ]:
cleaned = []
for name in names:
    this = re.sub('[Dd]ay$', '', name)
    cleaned.append(this)
cleaned

In [ ]:
# TODO : substitute these lines code with 1 line of code using list comprehension
# your code here

cleaned2 = ...
cleaned2


<a class="anchor" id="second-bullet"></a>
## 2 - Plotting with matplotlib and beyond
<br>
`matplotlib` is a very powerful `python` library for making scientific plots.

We will not focus too much on the internal aspects of `matplotlib` in today's lab. There are many excellent tutorials out there for `matplotlib`.  For example,
* [`matplotlib` homepage](https://matplotlib.org/)
* [`matplotlib` tutorial](https://github.com/matplotlib/AnatomyOfMatplotlib)

Conveying your findings convincingly is an absolutely crucial part of any analysis. Therefore, you must be able to write well and make compelling visuals.  Creating informative visuals is an involved process and we won't cover that in this lab.  However, part of creating informative data visualizations means generating *readable* figures.  If people can't read your figures or have a difficult time interpreting them, they won't understand the results of your work.  Here are some non-negotiable commandments for any plot:
* Label $x$ and $y$ axes
* Axes labels should be informative
* Axes labels should be large enough to read
* Make tick labels large enough
* Include a legend if necessary
* Include a title if necessary
* Use appropriate line widths
* Use different line styles for different lines on the plot
* Use different markers for different lines

There are other important elements, but that list should get you started on your way.

We will work with `matplotlib` and `seaborn` for plotting in this class.  `matplotlib` is a very powerful `python` library for making scientific plots.  `seaborn` is a little more specialized in that it was developed for statistical data visualization.  We will cover some `seaborn` later in class. In the meantime you can look at the [seaborn documentation](https://seaborn.pydata.org)

First, let's generate some data.

#### Let's plot some functions

We will use the following three functions to make some plots:

* Logistic function:
  \begin{align*}
    f\left(z\right) = \dfrac{1}{1 + be^{-az}}
  \end{align*}
  where $a$ and $b$ are parameters.
* Hyperbolic tangent:
  \begin{align*}
    g\left(z\right) = b\tanh\left(az\right) + c
  \end{align*}
  where $a$, $b$, and $c$ are parameters.
* Rectified Linear Unit:
  \begin{align*}
    h\left(z\right) =
    \left\{
      \begin{array}{lr}
        z, \quad z > 0 \\
        \epsilon z, \quad z\leq 0
      \end{array}
    \right.
  \end{align*}
  where $\epsilon < 0$ is a small, positive parameter.

You are given the code for the first two functions.  Notice that $z$ is passed in as a `numpy` array and that the functions are returned as `numpy` arrays.  Parameters are passed in as floats.

You should write a function to compute the rectified linear unit.  The input should be a `numpy` array for $z$ and a positive float for $\epsilon$.

In [ ]:
import numpy as np

def logistic(z: np.ndarray, a: float, b: float) -> np.ndarray:
    """ Compute logistic function
      Inputs:
         a: exponential parameter
         b: exponential prefactor
         z: numpy array; domain
      Outputs:
         f: numpy array of floats, logistic function
    """
    # your code here

def stretch_tanh(z: np.ndarray, a: float, b: float, c: float) -> np.ndarray:
    """ Compute stretched hyperbolic tangent
      Inputs:
         a: horizontal stretch parameter (a>1 implies a horizontal squish)
         b: vertical stretch parameter
         c: vertical shift parameter
         z: numpy array; domain
      Outputs:
         g: numpy array of floats, stretched tanh
    """
    # your code here

def relu(z: np.ndarray, eps: float = 0.01) -> np.ndarray:
    """ Compute rectificed linear unit
      Inputs:
         eps: small positive parameter
         z: numpy array; domain
      Outputs:
         h: numpy array; relu
    """
    # your code here

Now let's make some plots.  First, let's just warm up and plot the logistic function.

In [ ]:
x = np.linspace(-5.0, 5.0, 100) # Equally spaced grid of 100 pts between -5 and 5
f = logistic(x, 1.0, 1.0) # Generate data

In [ ]:
plt.plot(x, f)
plt.xlabel('x')
plt.ylabel('f')
plt.title('Logistic Function')
plt.grid(True)

#### Figures with subplots

Let's start thinking about the plots as objects. We have the `figure` object which is like a matrix of smaller plots named `axes`. You can use array notation when handling it.

In [ ]:
fig, ax = plt.subplots(1,1) # Get figure and axes objects

ax.plot(x, f) # Make a plot

# Create some labels
ax.set_xlabel('x')
ax.set_ylabel('f')
ax.set_title('Logistic Function')

# Grid
ax.grid(True)

Wow, it's *exactly* the same plot!  Notice, however, the use of `ax.set_xlabel()` instead of `plt.xlabel()`.  The difference is tiny, but you should be aware of it.  I will use this plotting syntax from now on.

What else do we need to do to make this figure better?  Here are some options:
* Make labels bigger!
* Make line fatter
* Make tick mark labels bigger
* Make the grid less pronounced
* Make figure bigger

Let's get to it.

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,6)) # Make figure bigger

# Make line plot
ax.plot(x, f, lw=4)

# Update ticklabel size
ax.tick_params(labelsize=24)

# Make labels
ax.set_xlabel(r'$x$', fontsize=24) # Use TeX for mathematical rendering
ax.set_ylabel(r'$f(x)$', fontsize=24) # Use TeX for mathematical rendering
ax.set_title('Logistic Function', fontsize=24)

ax.grid(True, lw=1.5, ls='--', alpha=0.75)

Notice:
* `lw` stands for `linewidth`.  We could also write `ax.plot(x, f, linewidth=4)`
* `ls` stands for `linestyle`.
* `alpha` stands for transparency.

The only thing remaining to do is to change the $x$ limits.  Clearly these should go from $-5$ to $5$.

In [ ]:
#fig.savefig('logistic.png')

# Put this in a markdown cell and uncomment this to check what you saved.
# ![](logistic.png)

#### Resources
If you want to see all the styles available, please take a look at the documentation.
* [Line styles](https://matplotlib.org/2.0.1/api/lines_api.html#matplotlib.lines.Line2D.set_linestyle)
* [Marker styles](https://matplotlib.org/2.0.1/api/markers_api.html#module-matplotlib.markers)
* [Everything you could ever want](https://matplotlib.org/2.0.1/api/lines_api.html#matplotlib.lines.Line2D.set_marker)

We haven't discussed it yet, but you can also put a legend on a figure.  You'll do that in the next exercise.  Here are some additional resources:
* [Legend](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.legend.html)
* [Grid](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.grid.html)

`ax.legend(loc='best', fontsize=24);`

<div class="exercise"><b>Exercise</b></div>

Plot the three activation-style functions below on the same axes:

- logistic function: $L(x; 1)$
- hyperbolic tangent: $\tanh(2x)$
- rectified linear unit: $\mathrm{ReLU}(x; 0.01)$

Use the same input range for all three functions, for example:

```python
x = np.linspace(-5, 5, 200)

In [ ]:
# your code here

<div class="exercise"><b>Exercise</b></div>

These figures look nice in the plot and it makes sense for comparison. Now let's put the 3 different figures in separate plots.

* Make a separate plot for each figure and line them up on the same row.

In [ ]:
# your code here


<div class="exercise"><b>Exercise</b></div>

* Make a grid of 2 x 3 separate plots, 3 will be empty. Just plot the functions and do not worry about cosmetics. We just want you ro see the functionality.

In [ ]:
# your code here


<a class="anchor" id="third-bullet"></a>
## 3 - Simple Linear Regression

Linear regression and its many extensions are a workhorse of the statistics and data science community, both in application and as a reference point for other models. Most of the major concepts in machine learning can be and often are discussed in terms of various linear regression models. Thus, this section will introduce you to building and fitting linear regression models and some of the process behind it, so that you can 1) fit models to data you encounter 2) experiment with different kinds of linear regression and observe their effects 3) see some of the technology that makes regression models work.


### Linear regression with a toy dataset
We first examine a toy problem, focusing our efforts on fitting a linear model to a small dataset with three observations.  Each observation consists of one predictor $x_i$ and one response $y_i$ for $i = 1, 2, 3$,

\begin{align*}
(x , y) = \{(x_1, y_1), (x_2, y_2), (x_3, y_3)\}.
\end{align*}

To be very concrete, let's set the values of the predictors and responses.

\begin{equation*}
(x , y) = \{(1, 2), (2, 2), (3, 4)\}
\end{equation*}

There is no line of the form $\beta_0 + \beta_1 x = y$ that passes through all three observations, since the data are not collinear. Thus our aim is to find the line that best fits these observations in the *least-squares sense*, as discussed in lecture.

<div class="exercise"><b>Exercise (optional practice)</b></div>

* Make two numpy arrays out of this data, x_train and y_train
* Check the dimensions of these arrays
* Try to reshape them into a different shape
* Make points into a scatterplot

In [ ]:
# your code here
x_train = ...
y_train = ...

In [ ]:
x_train.shape
y_train.shape

In [ ]:
x_train = x_train.reshape(3,1)
x_train.shape

In [ ]:
# Make a simple scatterplot



#### Formulae
Linear regression is special among the models we study because it can be solved explicitly. While most other models (and even some advanced versions of linear regression) must be solved itteratively, linear regression has a formula where you can simply plug in the data.

For the single predictor case it is:
    $$
      \beta_1 = \frac{\sum_{i=1}^n{(x_i-\bar{x})(y_i-\bar{y})}}{\sum_{i=1}^n{(x_i-\bar{x})^2}}\\
      \beta_0 = \bar{y} - \beta_1\bar{x}\
    $$
    
Where $\bar{y}$ and $\bar{x}$ are the mean of the y values and the mean of the x values, respectively.

###  Building a model from scratch
In this part, we will solve the equations for simple linear regression and find the best fit solution to our toy problem.

The snippets of code below implement the linear regression equations on the observed predictors and responses, which we'll call the training data set.  Let's walk through the code.

We have to reshape our arrrays to 2D. We will see later why.

<div class="exercise"><b>Exercise</b></div>

* make an array with shape (2,3)
* reshape it to a size that you want

In [ ]:
# your code here
xx = ...


In [ ]:
# Reshape to be a proper 2D array
x_train = x_train.reshape(x_train.shape[0], 1)
y_train = y_train.reshape(y_train.shape[0], 1)

print(x_train.shape)

In [ ]:
# first, compute means
y_bar = np.mean(y_train)
x_bar = np.mean(x_train)

# build the two terms
numerator = np.sum( (x_train - x_bar)*(y_train - y_bar) )
denominator = np.sum((x_train - x_bar)**2)

print(numerator.shape, denominator.shape) #check shapes

* Why the empty brackets? (The numerator and denominator are scalars, as expected.)

In [ ]:
#slope beta1
beta_1 = numerator/denominator

#intercept beta0
beta_0 = y_bar - beta_1*x_bar

print("The best-fit line is {0:3.2f} + {1:3.2f} * x".format(beta_0, beta_1))
print(f'The best fit is {beta_0}')

<div class="exercise"><b>Exercise</b></div>

Turn the code from the above cells into a function called `simple_linear_regression_fit`, that inputs the training data and returns `beta0` and `beta1`.

To do this, copy and paste the code from the above cells below and adjust the code as needed, so that the training data becomes the input and the betas become the output.

```python
def simple_linear_regression_fit(x_train: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    
    return
```

Check your function by calling it with the training data from above and printing out the beta values.

In [ ]:
# Your code here

* Let's run this function and see the coefficients

In [ ]:
x_train = np.array([1 ,2, 3])
y_train = np.array([2, 2, 4])

betas = simple_linear_regression_fit(x_train, y_train)

beta_0 = betas[0]
beta_1 = betas[1]

print("The best-fit line is {0:8.6f} + {1:8.6f} * x".format(beta_0, beta_1))

<div class="exercise"><b>Exercise</b></div>

* Do the values of `beta0` and `beta1` seem reasonable?
* Plot the training data using a scatter plot.
* Plot the best fit line with `beta0` and `beta1` together with the training data.

In [ ]:
# Your code here

The values of `beta0` and `beta1` seem roughly reasonable.  They capture the positive correlation.  The line does appear to be trying to get as close as possible to all the points.

<a class="anchor" id="additional-matrix-linear-regression"></a>
## Additional - Linear Regression as Matrix Geometry

This section is an **additional challenge**. It connects three views of the same least-squares problem:

1. **Algebra:** compute coefficients with the normal equation.
2. **Geometry:** view predictions as points in a column space.
3. **Pseudoinverse:** handle duplicate feature information when the usual inverse fails.

Helpful Wikipedia reading:

* [Linear regression](https://en.wikipedia.org/wiki/Linear_regression)
* [Ordinary least squares](https://en.wikipedia.org/wiki/Ordinary_least_squares)
* [Moore-Penrose inverse](https://en.wikipedia.org/wiki/Moore%E2%80%93Penrose_inverse)
* [Projection in linear algebra](https://en.wikipedia.org/wiki/Projection_%28linear_algebra%29)

### Additional Part 1 - From the scalar formula to matrix form

In the previous section, you fit a simple line to the same training data:

```python
x_train = np.array([1, 2, 3])
y_train = np.array([2, 2, 4])
```

and saved the scalar-form answer in `betas`.

Now your goal is to get the **same intercept and slope**, but using matrix notation.
For simple linear regression, each prediction is

$$
\hat y_i = \beta_0 + \beta_1x_i.
$$

The matrix form writes all predictions at once:

$$
\hat y = X\beta.
$$

For one feature, the design matrix has a column of 1s for the intercept:

$$
X =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix},
\qquad
\beta =
\begin{bmatrix}
\beta_0 \\
\beta_1
\end{bmatrix}.
$$

Least squares chooses the coefficients that minimize the squared prediction error:

$$
\hat\beta = \arg\min_\beta \|y - X\beta\|^2.
$$

When $X^T X$ is invertible, the closed-form solution is

$$
\hat\beta = (X^T X)^{-1}X^T y.
$$


In [ ]:
# Additional Part 1 Exercise: reproduce the scalar result using matrix form
# Use the same x_train and y_train from the previous section.

# TODO 1: Build the design matrix X_matrix.
# Hint: the first column should be all 1s, and the second column should be x_vector.
# X_matrix = ...

# TODO 2: Compute beta_matrix using the normal equation:
# beta_matrix = (X^T X)^(-1) X^T y
# beta_matrix = ...

# TODO 3: Compute predictions using the matrix equation y_hat = X beta.
# y_hat_matrix = ...

# After you finish the TODOs, uncomment these lines to compare your answer
# with the scalar formula from the previous section.
# print("Matrix-form beta_hat:", beta_matrix)
# print("Scalar-form betas:    ", betas)
# print("Do they match?", np.allclose(np.asarray(beta_matrix).reshape(-1), np.asarray(betas).reshape(-1)))
# print("Matrix-form predictions:", y_hat_matrix)


### Additional Part 2 - Projection when the inverse exists

The vector $X\hat\beta$ is the model's best prediction vector. But $X\hat\beta$ must live inside the space spanned by the columns of $X$.

So least squares can be viewed as a projection:

\begin{align*}
y \quad \longrightarrow \quad \hat y = X\hat\beta.
\end{align*}

If $y$ is not already in the column space of $X$, the model cannot hit it exactly. Instead, it finds the closest reachable point $\hat y$. The leftover error is the residual:

\begin{align*}
r = y - \hat y.
\end{align*}

At the least-squares solution, the residual is perpendicular to every column of $X$:

\begin{align*}
X^T r = 0.
\end{align*}

In [ ]:
# Additional Part 2: compute beta_hat, y_hat, and the residual explicitly
X_plane = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
y_vector = np.array([2.0, 1.0, 4.0])

beta_hat = np.linalg.inv(X_plane.T @ X_plane) @ X_plane.T @ y_vector
y_hat = X_plane @ beta_hat
residual = y_vector - y_hat

projection_matrix = X_plane @ np.linalg.inv(X_plane.T @ X_plane) @ X_plane.T
y_hat_from_projection = projection_matrix @ y_vector

print("X matrix:")
print(X_plane)
print("\nTrue target y:                  ", y_vector)
print("beta_hat from normal equation:  ", beta_hat)
print("Prediction y_hat = X @ beta_hat:", y_hat)
print("Projection matrix @ y:          ", y_hat_from_projection)
print("Residual y - y_hat:             ", residual)
print("X^T residual:                   ", X_plane.T @ residual)
print("Is residual perpendicular to the feature columns?", np.allclose(X_plane.T @ residual, 0))

In [ ]:
# Additional Part 2: visualize y, y_hat = X @ beta_hat, and the residual in 3-D
X_visual = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.28, 0.18],
])

# Choose a point on the prediction plane, then move away from the plane to create y.
beta_visual = np.array([2.25, 1.45])
y_hat_visual = X_visual @ beta_visual
plane_normal = np.cross(X_visual[:, 0], X_visual[:, 1])
plane_normal = plane_normal / np.linalg.norm(plane_normal)
y_visual = y_hat_visual + 1.85 * plane_normal

# Solve least squares and recover the closest point on the plane.
beta_hat_visual = np.linalg.inv(X_visual.T @ X_visual) @ X_visual.T @ y_visual
y_hat = X_visual @ beta_hat_visual
residual = y_visual - y_hat

print("Least-squares beta_hat:", beta_hat_visual)
print("Closest prediction X @ beta_hat:", y_hat)
print("Residual y - X @ beta_hat:", residual)

s_values = np.linspace(-0.35, 3.25, 2)
t_values = np.linspace(-0.35, 2.65, 2)
S, T = np.meshgrid(s_values, t_values)
plane_points = S[..., None] * X_visual[:, 0] + T[..., None] * X_visual[:, 1]

fig = plt.figure(figsize=(9.2, 7.2))
fig.patch.set_facecolor("white")
ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor("white")

ax.plot_surface(
    plane_points[:, :, 0],
    plane_points[:, :, 1],
    plane_points[:, :, 2],
    color="#60a5fa",
    alpha=0.23,
    shade=False,
    linewidth=0,
)

for s in np.linspace(-0.35, 3.25, 5):
    line = np.outer(np.full_like(t_values, s), X_visual[:, 0]) + np.outer(t_values, X_visual[:, 1])
    ax.plot(line[:, 0], line[:, 1], line[:, 2], color="#2563eb", alpha=0.23, linewidth=1.4)
for t in np.linspace(-0.35, 2.65, 5):
    line = np.outer(s_values, X_visual[:, 0]) + np.outer(np.full_like(s_values, t), X_visual[:, 1])
    ax.plot(line[:, 0], line[:, 1], line[:, 2], color="#2563eb", alpha=0.23, linewidth=1.4)

origin = np.zeros(3)
ax.quiver(*origin, *y_visual, color="#ef4444", alpha=0.34, linewidth=2.4, arrow_length_ratio=0.06)
ax.quiver(*origin, *y_hat, color="#10b981", alpha=0.46, linewidth=2.4, arrow_length_ratio=0.07)
ax.plot(
    [y_hat[0], y_visual[0]],
    [y_hat[1], y_visual[1]],
    [y_hat[2], y_visual[2]],
    color="#7c3aed",
    linewidth=5.5,
    solid_capstyle="round",
)

ax.scatter(*y_visual, s=175, color="#ef4444", edgecolor="white", linewidth=2.5, depthshade=False)
ax.scatter(*y_hat, s=175, color="#10b981", edgecolor="white", linewidth=2.5, depthshade=False)
ax.scatter(*origin, s=38, color="#111827", edgecolor="white", linewidth=1.2, depthshade=False)

marker_size = 0.22
in_plane_direction = X_visual[:, 0] / np.linalg.norm(X_visual[:, 0])
residual_direction = residual / np.linalg.norm(residual)
corner_a = y_hat + marker_size * in_plane_direction
corner_b = corner_a + marker_size * residual_direction
corner_c = y_hat + marker_size * residual_direction
ax.plot(
    [corner_a[0], corner_b[0], corner_c[0]],
    [corner_a[1], corner_b[1], corner_c[1]],
    [corner_a[2], corner_b[2], corner_c[2]],
    color="#7c3aed",
    linewidth=2.2,
)

ax.text(*(y_visual + np.array([0.08, 0.04, 0.10])), "$y$", color="#dc2626", fontsize=24)
ax.text(*(y_hat + np.array([0.16, -0.38, -0.17])), r"$\hat y = X\hat\beta$", color="#059669", fontsize=16)
ax.text(0.56, 1.90, 0.72, r"prediction plane: all $X\beta$", color="#1d4ed8", fontsize=12)
ax.text(*(y_hat + 0.52 * residual + np.array([-0.25, -0.08, 0.02])), "residual", color="#5b21b6", fontsize=13)

ax.view_init(elev=24, azim=-52)
ax.set_box_aspect((3.6, 3.0, 2.4))
ax.set_xlim(-0.35, 3.25)
ax.set_ylim(-0.45, 2.75)
ax.set_zlim(-0.15, 2.45)
ax.set_title("Least squares projection", fontsize=18, pad=14)
ax.set_axis_off()
plt.tight_layout()

### Additional Part 3 - When the inverse does not exist

The normal-equation formula uses $(X^T X)^{-1}$, so it only works when $X^T X$ is invertible.

A common failure case is duplicate feature information. Suppose one feature is just twice another feature:

\begin{align*}
x_2 = 2x_1.
\end{align*}

Then $x_2$ does not give the model a new direction. The model has two coefficient knobs, $\beta_1$ and $\beta_2$, but they control the same prediction direction.

The normal-equation inverse fails because the best prediction may be unique while the coefficients are not unique. The pseudoinverse, written $X^+$, is the backup tool:

\begin{align*}
\hat\beta = X^+y.
\end{align*}

If many coefficient vectors give the same best prediction, the pseudoinverse chooses the one with the smallest coefficient length $\|\beta\|$.

In [ ]:
# Additional Part 3: see the normal equation fail, then use the pseudoinverse
x1 = np.array([1, 2, 3, 4])
x2 = 2 * x1
X_singular = np.column_stack([np.ones_like(x1), x1, x2])
y_singular = np.array([2, 3, 5, 7])

print("X_singular:")
print(X_singular)
print("\nColumn relationship: x2 = 2 * x1")
print("Rank of X:", np.linalg.matrix_rank(X_singular))
print("Number of columns in X:", X_singular.shape[1])

XTX_singular = X_singular.T @ X_singular
print("\nX^T X:")
print(XTX_singular)
print("Rank of X^T X:", np.linalg.matrix_rank(XTX_singular))

try:
    beta_normal = np.linalg.inv(XTX_singular) @ X_singular.T @ y_singular
    print("Normal-equation solution:", beta_normal)
except np.linalg.LinAlgError as error:
    print("\nNormal-equation inverse failed:", error)
    print("Reason: the feature columns are dependent, so X^T X is singular.")

beta_pinv_singular = np.linalg.pinv(X_singular) @ y_singular
y_hat_singular = X_singular @ beta_pinv_singular
residual_singular = y_singular - y_hat_singular

print("\nUsing the pseudoinverse: beta_hat = X^+ y")
print("Pseudoinverse beta_hat:", beta_pinv_singular)
print("Prediction X @ beta_hat:", y_hat_singular)
print("Residual y - X @ beta_hat:", residual_singular)
print("Coefficient length ||beta_hat||:", np.linalg.norm(beta_pinv_singular))

### Additional Part 4 - Geometry when coefficients are not unique

The picture below separates two spaces:

**Prediction space:** This is where $X\beta$ lives. Even if columns are dependent, the model can still have a closest prediction $\hat y$.

**Coefficient space:** This is where $\beta$ lives. When columns are dependent, multiple coefficient vectors can create the same prediction.

In this visualization, $x_2 = 2x_1$, so

\begin{align*}
X\beta = \beta_1x_1 + \beta_2x_2 = (\beta_1 + 2\beta_2)x_1.
\end{align*}

The coefficient pairs $[2, 0]$, $[0, 1]$, and $[0.4, 0.8]$ all create the same prediction. The pseudoinverse chooses $[0.4, 0.8]$ because it is the smallest-norm choice among them.

In [ ]:
# Additional Part 4: visualize dependent columns and non-unique coefficients
X_dependent = np.array([
    [1.0, 2.0],
    [0.0, 0.0],
])
y_target = np.array([2.0, 1.25])

XTX = X_dependent.T @ X_dependent
beta_pinv = np.linalg.pinv(X_dependent) @ y_target
y_hat = X_dependent @ beta_pinv
residual = y_target - y_hat

beta_a = np.array([2.0, 0.0])
beta_b = np.array([0.0, 1.0])
beta_c = beta_pinv
candidate_betas = [beta_a, beta_b, beta_c]

print("X matrix with dependent columns:")
print(X_dependent)
print("\nX^T X:")
print(XTX)
print("Rank of X:", np.linalg.matrix_rank(X_dependent))
print("Rank of X^T X:", np.linalg.matrix_rank(XTX))

try:
    np.linalg.inv(XTX)
    print("X^T X is invertible.")
except np.linalg.LinAlgError as error:
    print("X^T X is not invertible because:", error)

print("\nPseudoinverse beta_hat:", beta_pinv)
print("Closest prediction y_hat:", y_hat)
print("Residual:", residual)
for beta in candidate_betas:
    print(f"beta = {beta} -> X @ beta = {X_dependent @ beta}")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
fig.patch.set_facecolor("white")

# Left: prediction space.
ax = axes[0]
ax.set_facecolor("#fcfcff")
ax.set_title("Prediction space", fontsize=16, pad=12)
ax.plot([-0.20, 2.75], [0, 0], color="#2563eb", linewidth=16, alpha=0.08, solid_capstyle="round")
ax.plot([-0.20, 2.75], [0, 0], color="#2563eb", linewidth=3, solid_capstyle="round")

col1 = X_dependent[:, 0]
col2 = X_dependent[:, 1]
ax.arrow(0, 0, col1[0], col1[1], head_width=0.055, length_includes_head=True, color="#1d4ed8", alpha=0.95)
ax.arrow(0, 0, col2[0], col2[1], head_width=0.055, length_includes_head=True, color="#1d4ed8", alpha=0.38)
ax.text(0.72, 0.13, "$x_1$", color="#1d4ed8", fontsize=14)
ax.text(1.78, 0.13, "$x_2 = 2x_1$", color="#1d4ed8", fontsize=14)
ax.text(0.95, -0.18, "all reachable predictions $X\\beta$", color="#1d4ed8", fontsize=12)

ax.scatter(*y_target, s=150, color="#ef4444", edgecolor="white", linewidth=2.5, zorder=5)
ax.scatter(*y_hat, s=150, color="#10b981", edgecolor="white", linewidth=2.5, zorder=5)
ax.plot([y_hat[0], y_target[0]], [y_hat[1], y_target[1]], color="#7c3aed", linewidth=5, solid_capstyle="round")
ax.text(y_target[0] + 0.07, y_target[1] + 0.02, "$y$", color="#dc2626", fontsize=19)
ax.text(y_hat[0] + 0.07, y_hat[1] - 0.12, "$\\hat y$", color="#059669", fontsize=18)
ax.text(y_hat[0] + 0.10, 0.62, "residual", color="#5b21b6", fontsize=13)

ax.set_xlim(-0.18, 2.70)
ax.set_ylim(-0.35, 1.55)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# Right: coefficient space.
ax = axes[1]
ax.set_facecolor("#fcfcff")
ax.set_title("Coefficient space", fontsize=16, pad=12)

theta = np.linspace(0, 2 * np.pi, 300)
for radius in [0.45, 0.89, 1.35, 2.0]:
    ax.plot(radius * np.cos(theta), radius * np.sin(theta), color="#d1d5db", linewidth=1, alpha=0.55)

beta2_values = np.linspace(-0.25, 1.25, 200)
beta1_values = 2.0 - 2.0 * beta2_values
ax.plot(beta1_values, beta2_values, color="#2563eb", linewidth=4, solid_capstyle="round")
ax.text(0.82, 0.49, "$\\beta_1 + 2\\beta_2 = 2$", color="#1d4ed8", fontsize=13, rotation=-35)

for beta, label, color, offset in [
    (beta_a, "$[2, 0]$", "#f97316", (0.05, 0.04)),
    (beta_b, "$[0, 1]$", "#f97316", (0.06, 0.04)),
    (beta_c, "pseudoinverse", "#10b981", (0.08, 0.07)),
]:
    ax.scatter(beta[0], beta[1], s=135, color=color, edgecolor="white", linewidth=2.5, zorder=5)
    ax.text(beta[0] + offset[0], beta[1] + offset[1], label, color=color, fontsize=12)

ax.arrow(0, 0, beta_c[0], beta_c[1], head_width=0.035, length_includes_head=True, color="#10b981", alpha=0.85, linewidth=2)
ax.text(0.10, 0.28, "minimum norm", color="#047857", fontsize=12, rotation=63)
ax.scatter(0, 0, s=40, color="#111827", edgecolor="white", linewidth=1.2, zorder=5)
ax.text(-0.10, -0.09, "0", color="#374151", fontsize=11)

ax.axhline(0, color="#cbd5e1", linewidth=1)
ax.axvline(0, color="#cbd5e1", linewidth=1)
ax.set_xlim(-0.32, 2.25)
ax.set_ylim(-0.22, 1.33)
ax.set_xlabel("$\\beta_1$", fontsize=13)
ax.set_ylabel("$\\beta_2$", fontsize=13)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

<div class="exercise"><b>Additional Reflection</b></div>

1. Why does the residual in Part 2 become perpendicular to the prediction plane?
2. Why can the prediction $\hat y$ be unique even when the coefficients $\beta$ are not unique?
3. Why is the pseudoinverse point the smallest-norm solution in coefficient space?
4. In real datasets, what kinds of features might accidentally contain duplicate or nearly duplicate information?

<a class="anchor" id="fourth-bullet"></a>
## 4 - Building a model with `sklearn`

Now that we can fit a small linear regression example from scratch, we will use `scikit-learn` on a real IoT dataset. `scikit-learn` expects:

* `X`: a 2-D feature matrix, with one row per observation and one column per feature.
* `y`: a 1-D target vector, with one value per observation.

The task for this lab is to predict home appliance energy use from environmental sensor measurements.


### Practice with `sklearn` and an IoT energy dataset

We will use `energydata_complete.csv`, an appliance energy prediction dataset collected from a house with temperature, humidity, weather, and light measurements. Each row is a 10-minute measurement window. The target variable is:

* `Appliances`: appliance energy use in Wh for that 10-minute window.

The prediction problem is realistic: a home IoT system can observe temperature, humidity, light, and outdoor weather, but appliance use depends on human behavior too. Do not expect perfect predictions.


In [ ]:
from pathlib import Path
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidate_paths = [
    Path("energydata_complete.csv"),
    Path("../../IoT_Dataset/energydata_complete.csv"),
    Path("../IoT_Dataset/energydata_complete.csv"),
    Path("../../../../COSMOS_Labs/IoT_Dataset/energydata_complete.csv"),
    Path("COSMOS_Labs/IoT_Dataset/energydata_complete.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find energydata_complete.csv. Put it in this notebook folder "
        "or in COSMOS_Labs/IoT_Dataset/."
    )

energy = pd.read_csv(data_path, parse_dates=["date"])
energy = energy.sort_values("date").reset_index(drop=True)
energy.head()


In [ ]:
print(f"Rows: {energy.shape[0]:,}")
print(f"Columns: {energy.shape[1]}")
print(f"Date range: {energy['date'].min()} to {energy['date'].max()}")
print(f"Missing values: {energy.isna().sum().sum()}")

energy[["date", "Appliances", "lights", "T1", "RH_1", "T_out", "RH_out", "Windspeed"]].head()


### Feature engineering

The columns named `T1` through `T9` are indoor temperature measurements from different rooms. The columns named `RH_1` through `RH_9` are relative humidity measurements. Outdoor weather is captured by variables such as `T_out`, `RH_out`, `Windspeed`, `Visibility`, and `Tdewpoint`.

We will add simple time features from the timestamp. We will exclude `rv1` and `rv2` because those are random variables included in the original dataset, not physical IoT sensor readings.


In [ ]:
energy["hour"] = energy["date"].dt.hour
energy["day_of_week"] = energy["date"].dt.dayofweek
energy["is_weekend"] = (energy["day_of_week"] >= 5).astype(int)

target = "Appliances"
feature_columns = [
    col for col in energy.columns
    if col not in ["date", target, "rv1", "rv2"]
]

print(f"Target: {target}")
print(f"Number of features: {len(feature_columns)}")
print(feature_columns)


In [ ]:
energy[[target] + feature_columns].describe().T[["mean", "std", "min", "max"]].head(12)


### Exploratory plots

Before fitting a model, inspect the target. Appliance energy use is often bursty: many time windows have low energy use, with occasional larger spikes.


In [ ]:
one_week = energy.iloc[:7 * 24 * 6]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(one_week["date"], one_week[target], linewidth=1)
axes[0].set_title("Appliance energy use during the first week")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Appliances energy use (Wh)")
axes[0].tick_params(axis="x", rotation=30)

axes[1].hist(energy[target], bins=40, edgecolor="white")
axes[1].set_title("Distribution of appliance energy use")
axes[1].set_xlabel("Appliances energy use (Wh)")
axes[1].set_ylabel("Count")

plt.tight_layout()


<div class="exercise"><b>Exercise</b></div>

Compute the correlation between each feature and `Appliances`. Which sensor variables seem most related to appliance energy use? Are the strongest correlations large or small?


In [ ]:
# TODO: inspect the top correlations. Try to explain why these features might matter.

### Train/test split

We will use a random train/test split so that we can compare linear regression and KNN regression in a simple way. For a real deployment that predicts future days from past days, a chronological split would be more appropriate.


In [ ]:
X = energy[feature_columns]
y = energy[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)


### Evaluation metrics

We will use three metrics:

* MAE: mean absolute error, in Wh. Lower is better.
* RMSE: root mean squared error, in Wh. Lower is better and it penalizes large mistakes more strongly.
* R²: fraction of target variance explained by the model. Higher is better; 1.0 is perfect.


In [ ]:
def regression_report(model_name, y_true, y_pred):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


<a class="anchor" id="fifth-bullet"></a>
## 5 - Linear regression for appliance energy prediction

Start with simple linear regression using one feature. `lights` is a reasonable first feature because lights often indicate occupancy or activity, which can be related to appliance use.

<div class="exercise"><b>Exercise</b></div>

Run the model using `lights`. Then try replacing `single_feature` with another feature from `feature_columns`, such as `T2`, `RH_1`, `T_out`, or `hour`. Does the score improve?


In [ ]:
single_feature = "lights" # TODO: change this to a different feature and see how the results change

simple_lr = LinearRegression()
simple_lr.fit(X_train[[single_feature]], y_train)

simple_pred = simple_lr.predict(X_test[[single_feature]])
simple_report = regression_report(f"Linear regression ({single_feature})", y_test, simple_pred)
simple_report


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.scatter(X_test[single_feature], y_test, s=12, alpha=0.25, label="Actual")

order = np.argsort(X_test[single_feature].to_numpy())
ax.plot(
    X_test[single_feature].to_numpy()[order],
    simple_pred[order],
    color="black",
    linewidth=2,
    label="Linear model",
)

ax.set_xlabel(single_feature)
ax.set_ylabel("Appliances energy use (Wh)")
ax.set_title(f"Simple linear regression using {single_feature}")
ax.legend()
plt.tight_layout()


### Multiple linear regression

Now use all of the physical sensor and time features. This model can combine weak signals from many sensors.

<div class="exercise"><b>Exercise</b></div>

Fit a multiple linear regression model and compare its metrics with the one-feature model. Is the model better? Why might the improvement still be limited?


In [ ]:
# your code here

In [ ]:
coef_table = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": multi_lr.coef_,
})
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table.sort_values("abs_coefficient", ascending=False).head(10)


The coefficient table is useful for debugging, but be careful: features are measured in different units, so the largest raw coefficient is not automatically the most important physical variable.


<a class="anchor" id="sixth-bullet"></a>
## 6 - K-nearest neighbors regression

KNN regression predicts a test point by finding similar training points and averaging their target values. Because KNN uses distances, feature scaling matters. We will use a `Pipeline` that first standardizes each feature and then applies KNN.

<div class="exercise"><b>Exercise</b></div>

Run the KNN model with `k = 5`. Then try changing `knn_k`. What happens when `k` is very small? What happens when `k` is large?


In [ ]:
knn_k = 5  # TODO: try 1, 3, 11, 21, or 51.

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=knn_k),
)
knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)
knn_report = regression_report(f"KNN regression (k={knn_k})", y_test, knn_pred)
knn_report

<div class="exercise"><b>Exercise</b></div>

Compare the models. Which model gives the lowest MAE? Which gives the best R²? If you were building an energy dashboard for a smart home, would you prefer the more interpretable linear model or the stronger KNN model?


In [ ]:
# your code here

### Reflection questions

1. Why is predicting appliance energy harder than predicting a controlled physics experiment?
2. What important information about the home is not directly measured by these sensors?
3. Why does KNN with `k = 1` usually fit the training data extremely well but generalize worse?
4. What other IoT sensors could improve this task?
